# CT4110 – Topic 8 Assignment
## Grover's Algorithm and Quantum Walks

This exercise has four tasks. In **Task 1** you build and run a three-qubit Grover search. In **Task 2** you investigate the optimal number of iterations. In **Task 3** you extend the search to two marked states. In **Task 4** you compare a classical random walk with a simple coined quantum walk.

Complete every `YOUR CODE HERE` section and every written answer. Run the notebook from top to bottom before submission. Keep the requested circuit drawings, tables, counts and plots visible in your saved notebook.

In [ ]:
# Course setup – run this cell first
try:
    import qiskit
    import qiskit_aer
except ImportError:
    %pip install -q qiskit qiskit-aer matplotlib pylatexenc

import math
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_histogram

simulator = AerSimulator()
SHOTS = 2048
print("Qiskit", qiskit.__version__)
print("Shots per measured circuit:", SHOTS)

## Grover's algorithm: roadmap

For $N=2^n$ candidates and $M$ marked states, Grover's algorithm begins in the equal superposition

$$|s\rangle=H^{\otimes n}|0^n\rangle.$$

One Grover iteration applies:

1. a phase oracle $O_f|x\rangle=(-1)^{f(x)}|x\rangle$;
2. the diffuser $D=2|s\rangle\langle s|-I$.

The oracle changes the sign of each marked amplitude. The diffuser then reflects all amplitudes about their mean. Repeating this pair rotates the state toward the marked subspace. When $M$ is known, a useful iteration estimate is

$$r\approx\left\lfloor\frac{\pi}{4}\sqrt{\frac{N}{M}}\right\rfloor.$$

Applying many more iterations is not better: the state can rotate past the marked subspace, reducing the success probability.

## Task 1 – Search for the marked state `111`

Build the phase oracle and diffuser as reusable gates. For three qubits, a multi-controlled $Z$ can be implemented by applying $H$ to the final qubit, then `mcx` with the other qubits as controls, then $H$ again.

Your oracle must change the phase of $|111\rangle$ and leave all other computational-basis states unchanged.

In [ ]:
# Task 1 checkpoint 1 – build the oracle and diffuser
def oracle_111():
    oracle = QuantumCircuit(3, name="Oracle 111")
    # Implement a phase flip of |111> using H, mcx and H.
    # YOUR CODE HERE
    return oracle.to_gate()


def diffuser(n_qubits):
    diffusion = QuantumCircuit(n_qubits, name="Diffuser")
    # 1. Apply H to all qubits.
    # 2. Apply X to all qubits.
    # 3. Implement a multi-controlled Z on |11...1>.
    # 4. Undo the X and H layers.
    # YOUR CODE HERE
    return diffusion.to_gate()


oracle_111().definition.draw("mpl")
diffuser(3).definition.draw("mpl")

# REQUIRED OUTPUT CHECK — write these checks yourself.
# 1. Assert that both returned objects act on three qubits.
# 2. Use Operator to assert that the oracle and diffuser are unitary.
# 3. Print a clear message when all checks pass.
# YOUR CODE HERE

In [ ]:
# Task 1 checkpoint 2 – verify the oracle phases
# Start from an equal superposition, apply only the oracle, and inspect the
# statevector. Store the result in oracle_state.
oracle_test = QuantumCircuit(3)
# YOUR CODE HERE

# Print each basis label and amplitude. Only 111 should have the opposite sign.
for index, amplitude in enumerate(oracle_state.data):
    print(f"{index:03b}: {amplitude.real:+.3f}{amplitude.imag:+.3f}j")

# REQUIRED OUTPUT CHECK — write these checks yourself.
# 1. Construct the expected eight signs: +1 for states 000 through 110
#    and -1 for state 111.
# 2. Remove the common 1/sqrt(8) amplitude scale from oracle_state.
# 3. Assert that the measured signs match the expected signs.
# 4. Print whether the oracle phase check passed.
# YOUR CODE HERE

In [ ]:
# Task 1 checkpoint 3 – construct, run and verify Grover search
grover_111 = QuantumCircuit(3, 3)

# Prepare |s>, apply two Grover iterations, then measure q0,q1,q2 into c0,c1,c2.
# Draw the circuit with output="mpl".
# YOUR CODE HERE

compiled_111 = transpile(grover_111, simulator)
result_111 = simulator.run(compiled_111, shots=SHOTS).result()
counts_111 = result_111.get_counts()
most_likely_111 = max(counts_111, key=counts_111.get)
success_probability_111 = counts_111.get("111", 0) / SHOTS

print("Counts:", counts_111)
print("Most likely state:", most_likely_111)
print("P(111):", round(success_probability_111, 4))
print("Target found:", most_likely_111 == "111")
plot_histogram(counts_111, title="Grover search for 111")

# REQUIRED OUTPUT CHECK — write these checks yourself.
# 1. Assert that the counts sum to SHOTS.
# 2. Assert that the circuit contains three measurements.
# 3. Assert that most_likely_111 is the marked state 111.
# 4. Assert that success_probability_111 is greater than 0.90.
# 5. Print a clear message when all checks pass.
# YOUR CODE HERE

## Task 2 – Find the best iteration count

For one marked item in an eight-item search space, compare 0 through 6 Grover iterations. This experiment demonstrates both amplification and over-rotation.

For each iteration count, build a **new** circuit, run it for `SHOTS`, and record the fraction of shots equal to `111`.

In [ ]:
# Task 2 checkpoint – measure success probability versus iteration count
iteration_counts = list(range(7))
success_probabilities = []

for repetitions in iteration_counts:
    experiment = QuantumCircuit(3, 3)
    # Prepare the equal superposition, repeat the oracle and diffuser exactly
    # "repetitions" times, measure, simulate, and append P(111).
    # YOUR CODE HERE
    pass

theory_estimate = math.floor((math.pi / 4) * math.sqrt(8 / 1))
best_observed = iteration_counts[int(np.argmax(success_probabilities))]

plt.figure(figsize=(7, 4))
plt.plot(iteration_counts, success_probabilities, "o-")
plt.axvline(theory_estimate, color="tab:red", linestyle="--", label="formula estimate")
plt.xlabel("Grover iterations")
plt.ylabel("Measured P(111)")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

print("Formula estimate:", theory_estimate)
print("Best observed iteration count:", best_observed)

# Explain why the curve rises and falls rather than increasing forever.
# ANSWER: [ ]

# REQUIRED OUTPUT CHECK — write these checks yourself.
# 1. Assert that iteration_counts and success_probabilities have equal length.
# 2. Assert that all success probabilities are between 0 and 1.
# 3. Assert that theory_estimate equals 2.
# 4. Assert that best_observed is the entry selected by np.argmax.
# 5. Assert that best_observed differs from theory_estimate by at most one.
# YOUR CODE HERE

## Task 3 – Search with two marked states

Now mark $|101\rangle$ and $|110\rangle$. A convenient oracle construction is:

1. for each target, apply $X$ to every qubit whose target bit is `0`;
2. apply the same multi-controlled phase flip used for $|111\rangle$;
3. undo those $X$ gates.

Be careful about Qiskit's displayed bit strings: the string `q2 q1 q0` is shown in counts, while gate methods number qubits as `q0`, `q1`, `q2`.

In [ ]:
# Task 3 checkpoint 1 – create a general phase oracle
def phase_oracle(targets, n_qubits):
    oracle = QuantumCircuit(n_qubits, name="Phase oracle")
    for target in targets:
        if len(target) != n_qubits or any(bit not in "01" for bit in target):
            raise ValueError("Each target must be an n-qubit binary string")

        # Convert the displayed string q(n-1)...q0 into Qiskit qubit indices.
        zero_qubits = [q for q, bit in enumerate(reversed(target)) if bit == "0"]

        # Map this target to |11...1>, phase-flip it, then undo the mapping.
        # YOUR CODE HERE

    return oracle.to_gate()


targets = ["101", "110"]
oracle_two = phase_oracle(targets, 3)
oracle_two.definition.draw("mpl")

# REQUIRED OUTPUT CHECK — write these checks yourself.
# 1. Use Statevector or Operator to verify that only 101 and 110 receive
#    a negative phase.
# 2. Assert that all six unmarked states retain a positive phase.
# 3. Print a clear message when both checks pass.
# YOUR CODE HERE

In [ ]:
# Task 3 checkpoint 2 – choose iterations, run and evaluate both targets
n_qubits = 3
N = 2 ** n_qubits
M = len(targets)
recommended_iterations = math.floor((math.pi / 4) * math.sqrt(N / M))

grover_two = QuantumCircuit(n_qubits, n_qubits)
# Prepare |s>, apply the recommended number of oracle–diffuser pairs, and measure.
# YOUR CODE HERE

compiled_two = transpile(grover_two, simulator)
counts_two = simulator.run(compiled_two, shots=SHOTS).result().get_counts()
marked_shots = sum(counts_two.get(target, 0) for target in targets)
marked_probability = marked_shots / SHOTS

print("Recommended iterations:", recommended_iterations)
print("Counts:", counts_two)
print("P(marked):", round(marked_probability, 4))
print("Both targets observed:", all(counts_two.get(target, 0) > 0 for target in targets))
plot_histogram(counts_two, title="Grover search with two marked states")

# Compare the two marked counts with the largest unmarked count. Explain why
# Grover amplifies a marked subspace rather than guaranteeing equal finite-shot counts.
# ANSWER: [ ]

# REQUIRED OUTPUT CHECK — write these checks yourself.
# 1. Assert that recommended_iterations equals 1.
# 2. Assert that the counts sum to SHOTS.
# 3. Assert that both marked states occur at least once.
# 4. Assert that marked_probability is greater than 0.90.
# 5. Print a clear message when all checks pass.
# YOUR CODE HERE

## Task 4 – Classical and quantum walks on an eight-position cycle

Use a cycle labelled by the three-bit positions `000` through `111`. Both walkers start at `000`, make exactly six jumps, wrap around the cycle, and use `SHOTS` experimental runs.

The quantum walk uses position qubits `q0`, `q1`, `q2` and coin qubit `q3`. Treat `q0` as the least-significant position bit. For one jump, apply a Hadamard to the coin and then the conditional shift

$$S|0,x\rangle=|0,(x-1)\bmod 8\rangle,\qquad S|1,x\rangle=|1,(x+1)\bmod 8\rangle.$$

Thus one complete jump is $U=S(H\otimes I_8)$. Prepare the initial coin state $(|0\rangle+i|1\rangle)/\sqrt{2}$ using `H` followed by `S`. Immediately before measurement, call `reverse_bits()` and use the returned circuit so the displayed position labels remain in big-endian order.

For the classical walk, every jump is independently $-1$ or $+1$ with equal probability. Its final position is

$$x_6=\left(\sum_{j=1}^{6}X_j\right)\bmod 8.$$

In [ ]:
# Task 4 checkpoint 1 – construct one Qiskit quantum-walk step
N_POSITIONS = 8
N_JUMPS = 6
q0, q1, q2 = 0, 1, 2
coin = 3

def increment_position(qc):
    # Add 1 modulo 8 only when the coin is 1.
    # YOUR CODE HERE
    pass

def decrement_position(qc):
    # Subtract 1 modulo 8 only when the coin is 0.
    # YOUR CODE HERE
    pass

def quantum_walk_step(qc):
    # Apply the Hadamard coin and then both conditional shifts.
    # YOUR CODE HERE
    pass

# REQUIRED OUTPUT CHECK — write these checks yourself.
# 1. Test increment_position for all eight starting positions with coin 1.
# 2. Assert that every result equals (position + 1) modulo 8.
# 3. Test decrement_position for all eight starting positions with coin 0.
# 4. Assert that every result equals (position - 1) modulo 8.
# 5. Explicitly check both wrap-around cases.
# 6. Use Operator to assert that quantum_walk_step is unitary.
# 7. Print a clear message when all checks pass.
# YOUR CODE HERE

In [ ]:
# Task 4 checkpoint 2 – build and run the six-jump quantum walk
quantum_walk = QuantumCircuit(4, 3)

# Prepare the specified coin state, apply exactly N_JUMPS walk steps,
# reverse the circuit bits, and measure only the remapped position wires.
# Store the reversed measured circuit back in quantum_walk.
# YOUR CODE HERE

quantum_walk.draw("mpl")

# Run SHOTS shots with RANDOM_SEED. Convert the counts to an eight-element
# integer array named quantum_position_counts and then create
# quantum_probabilities. Print both the counts and total probability.
# YOUR CODE HERE

# REQUIRED OUTPUT CHECK — write these checks yourself.
# 1. Assert that the circuit has four qubits and three classical bits.
# 2. Assert that it contains exactly three measurements.
# 3. Assert that quantum_position_counts has shape (8,).
# 4. Assert that its counts sum to SHOTS.
# 5. Assert that quantum_probabilities sums to 1.
# 6. Assert that positions 1, 3, 5 and 7 have zero probability.
# 7. Print a clear message when all checks pass.
# YOUR CODE HERE

In [ ]:
# Task 4 checkpoint 3 – simulate the classical walk
# Use np.random.default_rng(RANDOM_SEED) to simulate SHOTS walkers. Each
# walker starts at 0 and makes exactly N_JUMPS choices from {-1, +1}.
# Store the eight counts in classical_position_counts and the normalized
# values in classical_probabilities. Print both counts and total probability.
# YOUR CODE HERE

# REQUIRED OUTPUT CHECK — write these checks yourself.
# 1. Assert that classical_position_counts has shape (8,).
# 2. Assert that its counts sum to SHOTS.
# 3. Assert that classical_probabilities sums to 1.
# 4. Assert that positions 1, 3, 5 and 7 have zero probability.
# 5. Print a clear message when all checks pass.
# YOUR CODE HERE

In [ ]:
# Task 4 checkpoint 4 – compare the final distributions
# Create one grouped bar chart with the classical and quantum probabilities.
# Label every tick with both its integer and three-bit position. Include a
# title, axis labels, legend and a probability axis beginning at zero.
# YOUR CODE HERE

# REQUIRED OUTPUT CHECK — write these checks yourself.
# 1. Assert that both probability arrays have length 8.
# 2. Assert that both distributions sum to 1.
# 3. Assert that both distributions are zero at all odd positions.
# 4. Print the highest-probability position and probability for each walk.
# YOUR CODE HERE

# WRITTEN COMPARISON
# 1. Why are only even positions reachable after six jumps?
# ANSWER: [ ]
#
# 2. Which position has the highest probability in each distribution? Quote
#    both probabilities from your experiment.
# ANSWER: [ ]
#
# 3. Explain the difference using classical probability addition and quantum
#    constructive and destructive interference.
# ANSWER: [ ]

## Submission checklist

Before submitting, confirm that:

- all `YOUR CODE HERE` sections are complete;
- all written `[ ]` answers are replaced;
- the oracle phase check prints `True`;
- Task 1 identifies `111` as the most likely result;
- Task 2 includes the iteration-versus-success plot;
- Task 3 reports the combined probability of both marked states;
- both random-walk probability distributions sum to 1;
- the saved notebook contains all required circuit drawings and plots.